# Deploy a System-Assigned Managed Online Endpoint

Generate the scoring function and request, create a Microsoft Entra-authenticated endpoint, deploy the registered foundation model at zero traffic, invoke it directly, and promote the tested deployment to 100% traffic.

This uses the selected workspace-assets endpoint and leaves existing user-assigned endpoints unchanged. Its system-assigned identity is independent of the notebook caller selected by `USE_MANAGED_IDENTITY`.

**Source:** Adapted from [Azure/azureml-examples simple managed deployment](https://github.com/Azure/azureml-examples/blob/37c3572b3ceafdaaa90ee4503c920cfff899df1f/sdk/python/endpoints/online/managed/online-endpoints-simple-deployment.ipynb) and this repository's endpoint patterns, MIT License.

In [ ]:
from pathlib import Path
from azure.ai.ml import MLClient
from azure.identity import AzureCliCredential, ManagedIdentityCredential
from dotenv import dotenv_values
import ast
import json
import textwrap
from azure.ai.ml.entities import (
    CodeConfiguration,
    IdentityConfiguration,
    ManagedOnlineDeployment,
    ManagedOnlineEndpoint,
)
from azure.core.exceptions import ResourceNotFoundError

notebook_path = Path(globals().get("__vsc_ipynb_file__", Path.cwd() / "06_deploy_online_endpoint.ipynb"))
SUITE_ROOT = notebook_path.resolve().parents[1]
env = dotenv_values(SUITE_ROOT / ".env", encoding="utf-8-sig", interpolate=False)
SUBSCRIPTION_ID = (env.get("AZURE_SUBSCRIPTION_ID") or "").strip()
RESOURCE_GROUP = (env.get("AZURE_RESOURCE_GROUP") or "").strip()
WORKSPACE_NAME = (env.get("AZUREML_WORKSPACE_NAME") or "").strip()
if not all((SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME)):
    raise ValueError(f"Fill workspace coordinates in {SUITE_ROOT / '.env'}.")

use_managed_identity = (env.get("USE_MANAGED_IDENTITY", "true") or "").strip().lower()
if use_managed_identity not in {"true", "false"}:
    raise ValueError("USE_MANAGED_IDENTITY must be true or false in .env.")
credential = ManagedIdentityCredential(process_timeout=60) if use_managed_identity == "true" else AzureCliCredential(process_timeout=60)
ml_client = MLClient(credential, SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME)
ENDPOINT_NAME = f"{WORKSPACE_NAME[:14]}-workspace-sami-v2"
MODEL_NAME = "readiness-taxi-model"
INSTANCE_TYPE = "Standard_DS3_v2"
PUBLIC_ACCESS = "disabled"
DEPLOYMENT_NAME = "workspace"
ENVIRONMENT_NAME = "readiness-taxi-environment"
print(f"Workspace: {WORKSPACE_NAME}; new endpoint: {ENDPOINT_NAME}")

In [ ]:
generated_dir = SUITE_ROOT / "outputs/generated/foundations/online"
generated_code_dir = generated_dir / "code"
generated_code_dir.mkdir(parents=True, exist_ok=True)
(generated_code_dir / ".amlignore").write_text(
    "__pycache__/\n*.py[cod]\n", encoding="utf-8"
)
score_path = generated_code_dir / "score.py"
request_path = generated_dir / "request.json"
score_source = r'''
import json
import os
from pathlib import Path

import pandas as pd

_model = None


def init():
    global _model
    model_root = Path(os.environ["AZUREML_MODEL_DIR"])
    matches = list(model_root.rglob("model.json"))
    if len(matches) != 1:
        raise RuntimeError(f"Expected one model.json, found {len(matches)}")
    _model = json.loads(matches[0].read_text(encoding="utf-8"))


def run(raw_data):
    payload = json.loads(raw_data) if isinstance(raw_data, (str, bytes)) else raw_data
    input_data = payload.get("input_data", {})
    columns = input_data.get("columns")
    rows = input_data.get("data")
    if columns != _model["features"]:
        raise ValueError(f"Expected columns in this order: {_model['features']}")
    if not isinstance(rows, list) or not rows:
        raise ValueError("Request must contain at least one row")
    frame = pd.DataFrame(rows, columns=columns).apply(pd.to_numeric, errors="raise")
    predictions = _model["intercept"]
    for feature, coefficient in zip(
        _model["features"], _model["coefficients"], strict=True
    ):
        predictions = predictions + frame[feature] * coefficient
    return {"predictions": predictions.astype(float).tolist()}
'''
score_source = textwrap.dedent(score_source).lstrip()
ast.parse(score_source, filename=str(score_path))
score_path.write_text(score_source, encoding="utf-8")
request = {
    "input_data": {
        "columns": ["tripDistance", "passengerCount"],
        "data": [[2.5, 1], [7.0, 2]],
    }
}
request_path.write_text(json.dumps(request, indent=2) + "\n", encoding="utf-8")
print(f"Generated scoring script: {score_path}")
print(f"Generated request: {request_path}")

EXPECTED_BASE_IMAGE = "mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest"

registered_model = ml_client.models.get(MODEL_NAME, label="latest")
registered_environment = ml_client.environments.get(
    ENVIRONMENT_NAME,
    label="latest",
)
if registered_environment.image != EXPECTED_BASE_IMAGE:
    raise ValueError(
        f"Environment {registered_environment.name}:"
        f"{registered_environment.version} uses "
        f"{registered_environment.image}, expected {EXPECTED_BASE_IMAGE}. "
        "Rerun 03_register_environment.ipynb to register a new version."
    )
print(f"Model: {registered_model.name}:{registered_model.version}")
print(f"Environment: {registered_environment.name}:{registered_environment.version}")
print(f"Base image: {registered_environment.image}")

endpoint_definition = ManagedOnlineEndpoint(
    name=ENDPOINT_NAME,
    description="Readiness endpoint with a system-assigned identity",
    auth_mode="aad_token",
    identity=IdentityConfiguration(type="system_assigned"),
    public_network_access=PUBLIC_ACCESS,
    tags={"validation": "workspace-readiness", "purpose": "foundations"},
)
deployment_definition = ManagedOnlineDeployment(
    name=DEPLOYMENT_NAME,
    endpoint_name=ENDPOINT_NAME,
    model=registered_model,
    environment=registered_environment,
    code_configuration=CodeConfiguration(
        code=str(generated_code_dir),
        scoring_script="score.py",
    ),
    instance_type=INSTANCE_TYPE,
    instance_count=1,
)
print("Prepared only; run the next cell to create the endpoint.")

## Create the System-Assigned Endpoint

This creates or reuses only the selected endpoint name. The workspace prefix is limited to 14 characters to fit Azure's 32-character endpoint-name limit. No old endpoint is deleted or converted, and this notebook does not explicitly grant RBAC roles. Review the printed principal ID and required Storage/ACR access before deploying.

In [ ]:
try:
    endpoint = ml_client.online_endpoints.get(ENDPOINT_NAME)
except ResourceNotFoundError:
    endpoint = ml_client.online_endpoints.begin_create_or_update(endpoint_definition).result()
identity_type = endpoint.identity.type if endpoint.identity else None
if not identity_type or identity_type.replace("_", "").lower() != "systemassigned":
    raise ValueError(f"Endpoint {endpoint.name} has identity type {identity_type!r}; expected 'system_assigned'. Choose a new ENDPOINT_NAME.")
print(f"Endpoint: {endpoint.name}; system-assigned principal: {endpoint.identity.principal_id}")

## Deploy, Invoke, and Promote

This cell creates the billable `workspace` deployment on the workspace-assets endpoint. It invokes that deployment directly and only promotes traffic after the expected response is verified.

In [ ]:
deployment = ml_client.online_deployments.begin_create_or_update(deployment_definition).result()
if deployment.provisioning_state != "Succeeded":
    raise RuntimeError(f"Deployment state is {deployment.provisioning_state}")

response = ml_client.online_endpoints.invoke(
    endpoint_name=ENDPOINT_NAME,
    deployment_name=DEPLOYMENT_NAME,
    request_file=str(request_path),
)
body = json.loads(response)
if body.get("predictions") != [9.2, 20.2]:
    raise RuntimeError(f"Unexpected scoring response: {body}")
print(body)

endpoint = ml_client.online_endpoints.get(ENDPOINT_NAME)
endpoint.traffic = {DEPLOYMENT_NAME: 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print(f"Traffic promoted to {DEPLOYMENT_NAME}")


## Expected Result

The deployment reaches `Succeeded`, direct invocation returns `[9.2, 20.2]`, and the tested deployment receives 100% of endpoint traffic.

For an external-registry deployment, complete `08_register_assets_to_registry.ipynb` and then run `09_deploy_registry_online_endpoint.ipynb`. Model data collection is deferred.